In [4]:
#Langkah 1: Generate dan Eksplorasi Dataset
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
        'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

#buat transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

#suntikan pola: roit sering bersama selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

#print(f'Transaksi:\n{transaksi}')
print('Contoh Transaksi:', transaksi[:3])
print('\nJumlah Transaksi:', len(transaksi))

Contoh Transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]

Jumlah Transaksi: 50


In [5]:
#Langkah 2: One-Hot Encoding Transaksi
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)

df = pd.DataFrame(te_ary, columns=te.columns_)
#df.columns = df.columns.astype(str)

print(f"Shape df: {df.shape}")
print("Kolom:", df.columns.tolist())
print(df.head())

Shape df: (50, 10)
Kolom: [np.str_('Gula'), np.str_('Keju'), np.str_('Kopi'), np.str_('Mentega'), np.str_('Roti'), 'Selai', np.str_('Sereal'), np.str_('Susu'), np.str_('Teh'), np.str_('Telur')]
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [6]:
#Langkah 3: Cari Frequent Itemset dengan Apriori
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(f"\nItemset yang ditemukan ({len(freq_items)} itemset):")
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Itemset yang ditemukan (44 itemset):
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
#Langkah 4: Bentuk & Saring Aturan Asosiasi
from mlxtend.frequent_patterns import association_rules

def clean_itemset(itemset):
    return frozenset([str(item) for item in itemset])

freq_items['itemsets'] = freq_items['itemsets'].apply(clean_itemset)

if len(freq_items) > 0:
    rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
    rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
    print("\n=== Aturan Asosiasi ===")
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))
else:
    print("Tidak ada itemset yang ditemukan. Coba turunkan min_support.")


=== Aturan Asosiasi ===
         antecedents consequents  support  confidence      lift
9        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
14  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
12      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
13     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
10     (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
15   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Ringkasan Analisis Aturan Asosiasi1. Insight Utama (Aturan Terkuat)Hubungan Paling Kuat (Lift 2.38): {Keju, Teh} → {Telur}. Hubungan ini 2.38x lebih mungkin terjadi dibanding acak; mengindikasikan pola belanja "Bahan Sarapan/Omelet".Aturan dengan Kepastian Tertinggi (Confidence 100%): {Gula, Roti} → {Selai}. Seluruh pelanggan yang membeli Gula dan Roti pasti membeli Selai.Pola Paling Umum (Support 22%): {Roti} → {Selai}. Terjadi pada hampir 1 dari 4 transaksi, menunjukkan produk esensial yang sering dibeli bersama.2. Tabel Metrik KunciAturanLiftConfidenceSupport{Keju, Teh} → {Telur}2.3885.7%12%{Mentega, Selai} → {Kopi}1.9562.5%10%{Gula, Roti} → {Selai}1.92100%10%{Sereal} → {Mentega}1.8577.8%14%3. Implikasi StrategisPenataan Rak (Layouting): Kelompokkan produk komplementer (Keju-Teh-Telur dan Mentega-Selai-Kopi) dalam satu area "Sarapan" untuk memicu impulse buying.Promosi Bundling: Buat paket hemat "Sarapan Sehat" (Roti, Selai, Teh, Telur) atau paket "Kopi Pagi".Rekomendasi Digital: Gunakan algoritma cross-selling di platform online untuk menyarankan produk pelengkap (Contoh: "Pembeli Gula & Roti pasti membutuhkan Selai").Manajemen Stok: Prioritaskan ketersediaan stok untuk produk dengan Support tinggi (seperti Roti & Selai) karena merupakan penggerak utama transaksi.Catatan Analis:Support mengukur popularitas item.Confidence mengukur keandalan aturan.Lift > 1 menunjukkan hubungan positif yang signifikan antar produk.

In [8]:
#Langkah 5: Rekomender Sederhana dengan Content-Based Filtering
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
            'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [9]:
#Langkah 6: Bandingkan Kedua Pendekatan
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x)]

print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))
# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Berikut adalah versi yang lebih ringkas dan profesional:

Perbandingan & Strategi Rekomendasi
Perbandingan Metode:

Konsistensi: Keduanya merekomendasikan Selai untuk pembeli Roti.

Perbedaan: Association Rules fokus pada pola transaksi aktual, sedangkan Content-Based memperluas rekomendasi berdasarkan kemiripan kategori (Selai, Sereal, Susu).

Kapan Menggunakan:

Association Rules: Saat tersedia data transaksi yang cukup untuk melihat perilaku belanja.

Content-Based: Efektif untuk produk baru atau saat data transaksi masih minim.

Hybrid: Solusi optimal dengan menggabungkan keunggulan kedua metode.

Rekomendasi Hybrid untuk Roti:

Prioritas Utama: Selai (kesepakatan kedua metode).

Rekomendasi Sekunder: Sereal & Susu (kemiripan kategori).

Pelengkap Lain: Mentega & Telur (berdasarkan pola perilaku belanja).

Berikut adalah penyusunan ulang kesimpulan agar terlihat lebih segar namun tetap mempertahankan esensi pembelajaran praktikum Anda:

Kesimpulan Praktikum
Poin Utama Pembelajaran
Analisis Transaksi: Menerapkan algoritma Apriori untuk mengidentifikasi pola pembelian asosiatif, seperti kombinasi produk sarapan.

Persiapan Data: Melakukan transformasi dataset transaksi ke bentuk matriks biner menggunakan One-Hot Encoding agar siap diolah oleh algoritma.

Sistem Rekomendasi: Membangun model Content-Based Filtering yang memanfaatkan cosine similarity untuk menyarankan produk berdasarkan kemiripan kategori.

Evaluasi Strategis: Membandingkan efektivitas pendekatan berbasis perilaku (Association Rules) dengan pendekatan berbasis atribut produk (Content-Based).

Sintesis Temuan
Validasi Bisnis: Pola {Roti → Selai} terbukti signifikan melalui Association Rules, selaras dengan kebiasaan konsumsi riil pelanggan.

Perbedaan Karakteristik: Association Rules menangkap tren pembelian nyata, sementara Content-Based mengandalkan keterkaitan kategori, seperti pengelompokan produk bakery.

Potensi Hybrid: Implementasi sistem rekomendasi yang menggabungkan kedua metode ini memberikan hasil yang jauh lebih lengkap dan akurat dibanding penggunaan metode tunggal.

Evaluasi Kritis & Pengembangan
Keterbatasan: Analisis saat ini masih dibatasi oleh jumlah data yang sedikit (50 sampel), minimnya fitur atribut (hanya kategori), dan batasan jumlah item dalam setiap transaksi.

Arah Pengembangan:

Menguji skalabilitas Apriori pada dataset skala besar (ribuan transaksi).

Mengeksplorasi algoritma sistem rekomendasi alternatif yang lebih canggih untuk konteks e-commerce.

Mendalami arsitektur teknis untuk mengintegrasikan Association Rules dan Content-Based Filtering ke dalam sistem hybrid.